# 04 · ADHD shared pool (EEG Data for ADHD/Control Children)

This notebook reproduces **Table `tab:ieee-ranking`** of the paper and the ADHD numbers of Section *Cross-subject transfer in ADHD*: the ranking of MIL-CREDA and the seven comparators with the configuration picked by the label-free criterion cef and with the best test balanced accuracy (an upper bound), the Friedman and Holm-corrected Wilcoxon tests, Cliff's delta, and the paired bootstrap of MIL-CREDA's gain over its first phase.

**What runs.** Ten cross-subject units `ieee-r{r}h{h}` (rotation r of five subject folds, target-test half h). In each unit every method starts phase 2 from the same frozen phase-1 base and trains the same 100 random phase-2 settings of the shared pool (100 epochs, one seed per trial). cef picks one setting per method without labels; the oracle picks the best test BA. Training is done by the paper's own entrypoint, `MIL_CREDA_EEG.ieee_pool.run`; this notebook only drives it, joins the outputs and draws the table.

**Data.** The public Kaggle dataset [`daprosero/mi-tdah-dataset`](https://www.kaggle.com/datasets/daprosero/mi-tdah-dataset) (CC0), folder `TDAH/ieee` (121 children, 19 channels, 128 Hz). On Kaggle attach the dataset; elsewhere `repro.ieee_root()` downloads the 121 files (about 33 MB) with `kagglehub`. Preprocessing runs on the fly: 50 Hz notch, 4 s windows with 50 % overlap, at most 60 per subject, 1-45 Hz band-pass, common average reference; subject-wise stratified splits.

In [ ]:
import os, sys
from pathlib import Path

# src/ of the repository, from a local checkout or a Kaggle clone of
# https://github.com/Daprosero/Domain_Adaptation
here = Path.cwd()
if not (here / "src").exists() and not (here.parents[1] / "src").exists() and Path("/kaggle/working").exists():
    if not Path("/kaggle/working/Domain_Adaptation").exists():
        os.system("git clone --depth 1 https://github.com/Daprosero/Domain_Adaptation /kaggle/working/Domain_Adaptation")
    here = Path("/kaggle/working/Domain_Adaptation/MIL-CREDA-EEG/Notebooks")
root = next(p for p in (here, *here.parents) if (p / "src" / "MIL_CREDA_EEG").exists())
sys.path.insert(0, str(root / "src"))

from MIL_CREDA_EEG import repro
from MIL_CREDA_EEG.reproduce import stage04_adhd_pool as stage
print("repository:", root)
print("results:", stage.stage_dir())

## Parameters

Exactly those of the paper's 80 Kaggle jobs `ix-r{0-4}h{0,1}-s{0-7}` (commit `d7a4ef3e`, protocol v82): the `run.kwargs` of each job's `run-config.json` (`procs=4`, `shards=8`, `threads=1`, one unit and one shard) and the defaults of `ieee_pool.run` they leave in place (100 trials and 100 phase-2 epochs per method, phase-2 seed 1, mid-run snapshot of MIL-CREDA). The phase-1 bases are the paper's ten frozen seed-1 bases, stored in `Results/paper/04_adhd_pool/bases/`.

**Cost of the full run.** 80 CPU jobs (no GPU), 8000 trials; 90 CPU-hours of training in total, 10 to 23 minutes of wall time per job with 4 processes (24 job-hours in total, from the campaign's Kaggle logs). The paper ran the jobs in parallel over several Kaggle accounts. One account can run everything with `SPLIT = None`, sequentially, in about a day of CPU; or set `SPLIT = (k, n)` in n notebooks (one per account) to run the (unit, shard) pairs k, k+n, k+2n, ...

In [ ]:
print({k: v for k, v in stage.PARAMS.items()})
print(len(stage.UNITS), "work items (unit, shard), e.g.", stage.UNITS[:3], "...")

SPLIT = None  # or (k, n) to run portion k of n in one Kaggle account
SMOKE = os.environ.get("REPRO_SMOKE") == "1"   # one unit, one shard, one trial per method, 2 epochs
print("this run:", "smoke" if SMOKE else f"{len(repro.portion(stage.UNITS, SPLIT))} work items")

## Run

Each (unit, shard) writes `Results/paper/04_adhd_pool/raw/ix-r{r}h{h}-s{s}/`: one JSON with every trial's BA, AUC and kappa on the target-test half, and one `.npz` per trial with the outputs every later ADHD analysis reads (target and source probabilities and representations, the attention of every bag). A finished trial is never rerun, so an interrupted session resumes. The raw folder of the full campaign is about 1.9 GB and is not committed.

In [ ]:
root_ieee = repro.ieee_root()
print("IEEE data:", root_ieee)
done = stage.run(split=SPLIT, smoke=SMOKE)
print(done[-1] if done else "nothing to run")

## Merge

When the work ran in several accounts, copy every account's `raw/ix-*` folders into the same `Results/paper/04_adhd_pool/raw/` and run this cell once: it joins the 80 portions, condenses them to the per-trial fields the analysis needs (`summary/trials.csv`, `ptest.npz`, `units.json`, `pick_arrays.npz`, a few MB), recomputes the ranking and the tests (`ieee_pool_summary.json`) and the bootstrap (`ieee_bootstrap.json`). It refuses while a portion is missing. In smoke mode it condenses the smoke run into a scratch folder and leaves the stored summary untouched.

In [ ]:
import tempfile
if SMOKE:
    scratch = Path(tempfile.mkdtemp())
    sd = stage.condense(stage.smoke_dir(), scratch, partial=True)
    print("smoke condensed into", sd, sorted(p.name for p in sd.iterdir()))
elif SPLIT is None or all((stage.stage_dir() / "raw" / stage.job_name(u, s)).exists() for u, s in stage.UNITS):
    print("summary written to", stage.merge())
else:
    print("portion done; merge once every account's raw folders are copied here")

## Render

The table is drawn from the stored summary only (no training), as LaTeX and CSV under `Results/paper/04_adhd_pool/tables/`.

In [ ]:
import pandas as pd
out = stage.render()
print(out["table"])
display(pd.read_csv(out["csv"]))
display(pd.read_csv(out["bootstrap"]))

## Comparison with the paper

Every value the paper prints for this stage (Table `tab:ieee-ranking`, its caption and Section *Cross-subject transfer in ADHD*), next to the value obtained from the summary. A full rerun may differ slightly from run to run (thread scheduling, library versions); the stored summary reproduces the paper exactly.

In [ ]:
rows = repro.compare(stage.paper_checks())
print(sum(r["within"] for r in rows), "of", len(rows), "within tolerance")